# ฝึกโมเดลพื้นที่ผ่านได้จาก CVAT — รวมหลายคลิปเป็นโมเดลเดียว

ใช้ ZIP annotation จาก CVAT (Segmentation mask แบบไม่รวมภาพ) คู่กับวิดีโอต้นฉบับของแต่ละคลิป รองรับตั้งแต่ 1 คู่จนถึงหลายคู่

**ตั้งชื่อ ZIP ให้ตรงกับวิดีโอ:** `part1.zip` + `part1.mov`, `part2.zip` + `part2.mov` ฯลฯ เปลี่ยนเฉพาะชื่อ ZIP ด้านนอก ไม่เปลี่ยนชื่อไฟล์ mask ข้างใน

1. เลือก **Runtime → Change runtime type → T4 GPU** ถ้ามีให้ใช้ แล้วกด Save
2. กด ▶ ของช่องโค้ดตามลำดับ 1–5 รอแต่ละช่องเสร็จก่อนกดช่องถัดไป
3. ช่อง 2 อัปโหลดทุกคู่พร้อมกัน ช่อง 3 รวมชุดข้อมูล ช่อง 4 ฝึกครั้งเดียว และช่อง 5 ดาวน์โหลดโมเดลเดียว

Colab มีโหมดใช้ฟรี แต่ไม่ได้รับประกัน GPU ว่าง ไม่ต้องสมัคร Pro เพื่อเปิด notebook นี้ ถ้าได้ CPU โค้ดจะทดลองฝึก 3 epochs และอาจช้า; GPU ฝึกต้นแบบ 20 epochs

ค่า loss และภาพตัวอย่างมาจากชุดฝึก ไม่ใช่ชุดทดสอบ part1–4 ที่บันทึกต่อกันยังเป็น recording group เดียว ต้องมีการบันทึกอิสระสำหรับประเมินพื้นที่หรือแสงใหม่


## 1 — ติดตั้งเครื่องมือ

ช่องนี้โหลดโค้ดจาก GitHub และติดตั้ง dependency ใน Colab ไม่เปลี่ยน venv บน Ubuntu ของคุณ


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/husk')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'work',
                    'https://github.com/jorjor140900/husk.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO / 'requirements-segmentation.txt')], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# Refresh this helper if the same runtime used the older single-clip notebook.
import importlib
importlib.invalidate_caches()
if 'tools.prepare_cvat_dataset' in sys.modules:
    importlib.reload(sys.modules['tools.prepare_cvat_dataset'])
import torch
print('พร้อมใช้งาน | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — ใช้ CPU ได้แต่ช้ากว่า')


## 2 — อัปโหลด ZIP และวิดีโอทุกคู่

กด Choose Files แล้วเลือกไฟล์พร้อมกัน (Ctrl+คลิก) ถ้าฝึกรวม part1–4 ให้เลือก **8 ไฟล์**:

- `part1.zip`, `part1.mov`
- `part2.zip`, `part2.mov`
- `part3.zip`, `part3.mov`
- `part4.zip`, `part4.mov`

ใช้ ZIP annotation จาก CVAT กับวิดีโอต้นฉบับ รองรับ `.mov`, `.mp4`, `.avi` ไม่ใส่ ZIP โมเดลที่เคยฝึกแล้ว ช่องนี้ยังใช้กับ part1 คู่เดียวได้


In [ ]:
from datetime import datetime, timezone
from google.colab import files
from tools.prepare_cvat_dataset import match_clip_pairs

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
INPUTS = Path('/content/husky-inputs') / STAMP
INPUTS.mkdir(parents=True)
os.chdir(INPUTS)
try:
    uploaded = files.upload()
finally:
    os.chdir(REPO)
paths = [INPUTS / name for name in uploaded]
PAIRS = match_clip_pairs(paths)
del uploaded
for annotations, video in PAIRS:
    print('คู่ที่ใช้:', annotations.name, '+', video.name)
print('รวม', len(PAIRS), 'คลิป — จะฝึกรวมเป็นโมเดลเดียว')


## 3 — รวมภาพกับคำตอบจากทุกคลิป แล้วตรวจภาพรวม

สีเขียว=ผ่านได้ สีแดง=ห้ามผ่าน ส่วนมืด=ไม่ label ซึ่งไม่ใช้คำนวณ loss
ภาพแต่ละคลิปแยกโฟลเดอร์กัน แม้เลขเฟรมซ้ำก็ไม่ทับกัน แต่มี manifest รวมสำหรับฝึกโมเดลเดียว
ตรวจว่าขอบ label ตรงกับเสาและพื้นจริงก่อนกดช่องฝึก หากไม่ตรงให้ตรวจวิดีโอและเลขเฟรมก่อน


In [ ]:
from IPython.display import Image, display
from tools.prepare_cvat_dataset import prepare_collection

RUN = Path('/content/husky-runs') / STAMP
DATASET = RUN / 'dataset'
summary = prepare_collection(PAIRS, DATASET, 'front-building-session-01')
print('รวม', summary['clip_count'], 'คลิป ได้', summary['samples'], 'คู่ภาพ/คำตอบ')
for clip in summary['clips']:
    print(clip['source_video'], ':', clip['samples'], 'ภาพ | เฟรม', clip['source_frame_indices_zero_based'])
display(Image(filename=str(DATASET / 'contact-sheet.png'), width=1100))
print('ภาพตรวจครบทุกเฟรมของแต่ละคลิปอยู่ใน DATASET / clips / clip_NN / contact-sheet.png')


## 4 — ฝึก SegFormer-B0 ครั้งเดียวจากข้อมูลรวม

โหลด pretrained model สาธารณะจาก Hugging Face แล้ว fine-tune ด้วยข้อมูลทุกคลิปในชุดรวม ไม่วนฝึกแยกคลิปหรือสร้างสี่โมเดล

ถ้าเคยฝึกเฉพาะ part1 แล้ว ให้เก็บโมเดลเดิมไว้ รอบนี้ฝึก pretrained ใหม่ด้วย part1–4 รวมกัน และได้โมเดลรุ่นใหม่หนึ่งตัว โค้ดนี้ไม่ได้ resume โมเดล part1 เดิมโดยอัตโนมัติ

รอจนเห็น **Model saved to ...** โมเดลบันทึก checkpoint เมื่อจบแต่ละ epoch ค่า training loss ไม่ใช่ accuracy หากดาวน์โหลด pretrained ไม่สำเร็จจะหยุดพร้อมข้อผิดพลาด ไม่ใช้โมเดลสุ่มแทน


In [ ]:
TRAIN_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
MODEL = RUN / ('train-' + TRAIN_STAMP) / 'husky-segformer'
EPOCHS = 20 if torch.cuda.is_available() else 3
HEIGHT, WIDTH = (384, 640) if torch.cuda.is_available() else (256, 448)
print('เริ่มฝึก', EPOCHS, 'epochs บน', 'GPU' if torch.cuda.is_available() else 'CPU')
subprocess.run([
    sys.executable, '-m', 'tools.train_segmentation',
    '--dataset', str(DATASET), '--output', str(MODEL),
    '--epochs', str(EPOCHS), '--batch-size', '2', '--device', 'auto',
    '--height', str(HEIGHT), '--width', str(WIDTH),
], cwd=REPO, check=True)
for example in sorted((MODEL / 'training_previews').glob('*.jpg')):
    display(Image(filename=str(example), width=1100))
print('ตัวอย่างข้างบนมาจากชุดฝึก ยังไม่ใช่ผลทดสอบอิสระ')


## 5 — ดาวน์โหลดโมเดลกลับเครื่อง

ดาวน์โหลดจาก Colab โดยตรง ไม่ต้องดาวน์โหลด artifact จาก Codex เก็บ ZIP นี้ไว้ เพราะไฟล์ใน runtime ของ Colab อาจหายเมื่อ session สิ้นสุด


In [ ]:
import shutil

for name in ('config.json', 'preprocessor_config.json', 'model.safetensors', 'training_report.json'):
    if not (MODEL / name).is_file():
        raise FileNotFoundError('ยังไม่มีโมเดลครบชุด: ' + name + ' — ตรวจช่องฝึกก่อน')
archive = shutil.make_archive(str(RUN / 'husky-segformer'), 'zip',
                              root_dir=MODEL.parent, base_dir=MODEL.name)
print('ดาวน์โหลดโมเดล:', Path(archive).name)
files.download(archive)


## ใช้โมเดลกับ webcam บน Ubuntu หลังดาวน์โหลดเสร็จ

เปิด Terminal บน Ubuntu ใน repository ที่เคยเปิดกล้องได้ คำสั่งติดตั้งด้านล่างใช้ PyTorch CPU และคง OpenCV แบบ GUI ที่คุณใช้อยู่ ใช้โฟลเดอร์โมเดลใหม่เมื่อเปลี่ยนรุ่น

```bash
cd ~/husk-live
git pull --ff-only
source .venv/bin/activate
python -m pip install --index-url https://download.pytorch.org/whl/cpu 'torch==2.6.0'
python -m pip install 'transformers==4.49.0' 'safetensors>=0.4.3,<1' 'Pillow>=10,<12'
mkdir -p models
unzip ~/Downloads/husky-segformer.zip -d models
python RoadDetection.py --camera 0 --show --config config.example.json --model models/husky-segformer --device cpu
```

ถ้า browser บันทึก ZIP ไว้ตำแหน่งอื่น ให้แก้ path ของ unzip ให้ตรง **เก็บทั้งโฟลเดอร์โมเดล** รวม config และ preprocessor ไม่ใช้เฉพาะไฟล์ weights

โค้ดใช้ mask จากโมเดลแทน color/Canny แล้วใช้ตัวสร้าง reference เดิม ยังไม่มี ROS หรือคำสั่งควบคุมหุ่น ข้อมูล part1 ชุดเดียวใช้ทดลองกระบวนการได้ แต่ต้องเพิ่มภาพ/การบันทึกอิสระก่อนสรุปว่าแยกเสา เงา หรือพรมได้แม่น
